# SHIELD-ID — Layer 1: Detector de texto gerado por LLM (Colab GPU)

**O que faz:** fine-tune de um transformer pré-treinado (rule 05 — nunca do zero) como classificador
binário: **texto de documento humano (0)** vs **texto gerado por LLM (1)**. Avalia com o **protocolo
cross-generator** (treina em {A,B}, testa no gerador held-out C) e reporta o **robustness delta** + **FPR
desagregado** — não um número in-distribution.

> **Honestidade (M1/D5):** os números são *medidos*, com metodologia. Sem GPU não roda — por isso é aqui.
> Runtime → **Change runtime type → GPU** antes de começar.


In [ ]:
# 1) Dependências (torch já vem no Colab GPU)
!pip -q install 'transformers>=4.40' 'datasets>=2.19' scikit-learn


In [ ]:
# 2) GPU?
import torch; print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


In [ ]:
# 3) Config (rule 32 — tudo aqui, nada hardcoded na lógica)
CFG = dict(
    base_model='roberta-base',      # troque por um detector mais forte se quiser
    max_length=512, lr=2e-5, epochs=3, batch_size=16, seed=42,
    fpr_target=0.05,                # alvo de FPR p/ o ponto de operação (a TESTAR — D5)
    train_generators=['chatgpt'],   # geradores no treino
    held_out_generator='gpt4',      # gerador NUNCA visto no treino (I4/D8) — veja a célula de dados
)


## 4) Dados — o seu insumo
Formato alvo: lista de `{text, label(0/1), generator, segment}`.
- **Início rápido (abaixo):** carrega o **HC3** (humano vs ChatGPT) — real, confiável. É **1 gerador**,
  então o cross-generator fica limitado; serve pra validar o pipeline inteiro.
- **Cross-generator de verdade:** adicione um **2º gerador** (ex.: RAID/M4, ou gere com outra API) marcado
  como `held_out_generator`. Onde plugar está sinalizado com `# >>> ADICIONE`.
- **Sem PII real (I2):** o controle humano deve ser corpus público/sintético.


In [ ]:
from datasets import load_dataset
def load_records(max_per_class=2000):
    recs=[]
    # --- humano vs ChatGPT (HC3) ---  (ajuste se o schema do dataset mudar)
    ds = load_dataset('Hello-SimpleAI/HC3','all', split='train')
    h=c=0
    for row in ds:
        dom = row.get('source','unknown')   # usamos o domínio como 'segment' proxy (fairness precisa de segmentos reais)
        for t in (row.get('human_answers') or []):
            if t and h<max_per_class: recs.append(dict(text=t,label=0,generator='human',segment=dom)); h+=1
        for t in (row.get('chatgpt_answers') or []):
            if t and c<max_per_class: recs.append(dict(text=t,label=1,generator='chatgpt',segment=dom)); c+=1
    # >>> ADICIONE AQUI o gerador held-out (ex.: gpt4) p/ cross-generator real:
    #   for t in textos_gpt4: recs.append(dict(text=t,label=1,generator='gpt4',segment=...))
    print('registros:',len(recs),'| geradores:',set(r['generator'] for r in recs))
    return recs
records = load_records()


In [ ]:
# 5) Split cross-generator (recusa held-out no treino — rule 05/I4)
def split_xgen(rows, train_gens, held):
    assert held not in train_gens, 'held-out no treino = circularidade (rule 05/I4)'
    control=[r for r in rows if r['label']==0]
    train=[r for r in rows if r['generator'] in train_gens]+control
    test =[r for r in rows if r['generator']==held]+control
    return train, test
train, test = split_xgen(records, CFG['train_generators'], CFG['held_out_generator'])
print('treino:',len(train),'| held-out+controle:',len(test),'(0 se o held-out não tem dados — adicione!)')


In [ ]:
# 6) Detector (espelha src/shield_id/layers/layer1_detection) — fine-tune de modelo pré-treinado (rule 05)
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader, TensorDataset
tok = AutoTokenizer.from_pretrained(CFG['base_model'])
model = AutoModelForSequenceClassification.from_pretrained(CFG['base_model'], num_labels=2).cuda()
def enc(texts): return tok(list(texts),truncation=True,max_length=CFG['max_length'],padding=True,return_tensors='pt')
torch.manual_seed(CFG['seed'])
e=enc([r['text'] for r in train]); y=torch.tensor([r['label'] for r in train])
dl=DataLoader(TensorDataset(e['input_ids'],e['attention_mask'],y),batch_size=CFG['batch_size'],shuffle=True)
opt=torch.optim.AdamW(model.parameters(),lr=CFG['lr']); model.train()
for ep in range(CFG['epochs']):
    tot=0
    for ids,m,yy in dl:
        opt.zero_grad(); out=model(input_ids=ids.cuda(),attention_mask=m.cuda(),labels=yy.cuda())
        out.loss.backward(); opt.step(); tot+=out.loss.item()
    print(f'epoch {ep+1}/{CFG["epochs"]} loss={tot/len(dl):.4f}')


In [ ]:
# 7) Avaliação cross-generator + FPR desagregado (honesto: curvas/delta, não um ponto)
import numpy as np
def proba(texts):
    model.eval(); out=[]
    with torch.no_grad():
        for i in range(0,len(texts),64):
            b=enc(texts[i:i+64]); lo=model(input_ids=b['input_ids'].cuda(),attention_mask=b['attention_mask'].cuda()).logits
            out += torch.softmax(lo,-1)[:,1].cpu().tolist()
    return out
def pr_at_fpr(s,y,fpr):
    legit=sorted([si for si,yi in zip(s,y) if yi==0],reverse=True)
    thr=legit[min(len(legit)-1,int(fpr*len(legit)))] if legit else 1.0
    tp=sum(si>=thr and yi==1 for si,yi in zip(s,y)); fn=sum(si<thr and yi==1 for si,yi in zip(s,y))
    fp=sum(si>=thr and yi==0 for si,yi in zip(s,y)); tn=sum(si<thr and yi==0 for si,yi in zip(s,y))
    rec=tp/(tp+fn) if tp+fn else 0; fprr=fp/(fp+tn) if fp+tn else 0; return thr,rec,fprr
# in-distribution (treino) vs cross-generator (held-out)
si=proba([r['text'] for r in train]); yi=[r['label'] for r in train]
sc=proba([r['text'] for r in test]);  yc=[r['label'] for r in test]
_,rec_in,_=pr_at_fpr(si,yi,CFG['fpr_target'])
_,rec_xg,fpr_xg=pr_at_fpr(sc,yc,CFG['fpr_target'])
print(f'in-distribution recall@FPR={CFG["fpr_target"]}: {rec_in:.3f}')
print(f'CROSS-GENERATOR (held-out {CFG["held_out_generator"]}) recall: {rec_xg:.3f} | FPR: {fpr_xg:.3f}')
print(f'>>> ROBUSTNESS DELTA (a manchete): {(rec_xg-rec_in)*100:.1f} pp')
if not any(r['generator']==CFG['held_out_generator'] for r in test):
    print('AVISO: held-out sem dados -> número é IN-DISTRIBUTION, não cross-generator (I4). Adicione o 2º gerador.')


In [ ]:
# 8) Salvar modelo (model-card sem métrica — no repo, o eval-independent certifica isolado: M5/D4)
model.save_pretrained('artifacts/text-detector'); tok.save_pretrained('artifacts/text-detector')
print('modelo salvo em artifacts/text-detector  — baixe e plugue na FastAPI /verify (US-009)')
